## **AdaBoost Hyperparameters**

### **Topic Roadmap**

**1. Prepare a non-linear classification dataset**

**2. Compare learning rate and estimator count**

**3. Tune the weak learner**

**4. Evaluate the selected configuration**

**5. Key revision notes**

## **1. Dataset**

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

RANDOM_STATE = 42
X, y = make_circles(n_samples=1000, factor=0.25, noise=0.2, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## **2. Learning Rate and Number of Estimators**

A smaller learning rate usually requires more weak learners. The product of these settings controls the total additive capacity.

In [2]:
results = []
for learning_rate in [0.05, 0.2, 0.8]:
    for n_estimators in [50, 150, 300]:
        model = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            n_estimators=n_estimators, learning_rate=learning_rate,
            random_state=RANDOM_STATE
        )
        model.fit(X_train, y_train)
        results.append({"learning_rate": learning_rate, "n_estimators": n_estimators, "test_accuracy": model.score(X_test, y_test)})
pd.DataFrame(results).sort_values("test_accuracy", ascending=False)

,learning_rate,n_estimators,test_accuracy
3,0.20,50,0.970
7,0.80,150,0.965
4,0.20,150,0.965
6,0.80,50,0.965
5,0.20,300,0.965
8,0.80,300,0.965
2,0.05,300,0.960
1,0.05,150,0.810
0,0.05,50,0.610


## **3. Tune Weak-Learner Depth**

The base tree depth controls the complexity of each boosting step. A small grid makes the interaction with the ensemble visible.

In [3]:
param_grid = {
    "estimator__max_depth": [1, 2, 3],
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.05, 0.2, 0.5],
}
search = GridSearchCV(
    AdaBoostClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        random_state=RANDOM_STATE
    ),
    param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
search.fit(X_train, y_train)
print(search.best_params_)
print(f"Best CV accuracy: {search.best_score_:.3f}")

{'estimator__max_depth': 3, 'learning_rate': 0.2, 'n_estimators': 200}
Best CV accuracy: 0.957


In [4]:
print(f"Test accuracy: {search.best_estimator_.score(X_test, y_test):.3f}")

Test accuracy: 0.970


### **Key Revision Notes**

- `n_estimators` controls the number of additive weak learners.
- `learning_rate` shrinks each learner contribution.
- Weak-learner depth controls the complexity added at each step.
- Tune these parameters together rather than interpreting one in isolation.